# Benchmark Vintern 1B + 3B — 355 keyframe AIC

Hai model cung kien truc `InternVLChatModel`, cung ghim thu vien, chay noi tiep
trong mot luot. Bang benchmark hien chi co 2 model do that; luot nay them 2 nua.

`InternVLChatModel` thieu `all_tied_weights_keys` ma transformers 4.5x+ doi hoi,
nen phai cai ban cu hon -- KHONG dung moc `>=4.51,<5` cua Qwen.

Thu tu cell: chay -> nen/luu -> kiem. Cell kiem dat SAU cell nen vi kernel ERROR
lam Kaggle xoa sach /kaggle/working (da mat 1 luot GPU vi thu tu nguoc).

In [ ]:
import os

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import torch

print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG CO')
assert torch.cuda.is_available(), 'Chua bat GPU'


In [ ]:
# Moc CU HON Qwen: Vintern can transformers truoc khi all_tied_weights_keys
# thanh bat buoc. Ghim ca timm/einops vi InternVL goi toi qua trust_remote_code.
!pip install -q "transformers>=4.37,<4.50" accelerate bitsandbytes timm einops sentencepiece
import transformers
print('transformers:', transformers.__version__)
assert transformers.__version__ < '4.50', f'pip keo nham ban {transformers.__version__}'


In [ ]:
import subprocess, sys
from pathlib import Path

REPO = 'https://github.com/lolizabrett-byte/Multimodal-Agentic-Retrieval-Engine.git'
NHANH = 'research/vlm-prompting'
DICH = Path('/kaggle/working/repo')

# Kaggle giu /kaggle/working giua cac version, nen "clone neu chua co" se dung
# code cu cua lan chay truoc. Da mat mot luot vi vay: adapter moi khong duoc goi,
# loi cu lap lai y het. Xoa roi clone lai moi lan.
import shutil
if DICH.exists():
    shutil.rmtree(DICH)
subprocess.run(['git', 'clone', '--depth', '1', '-b', NHANH, REPO, str(DICH)], check=True)

PKG = DICH / 'system1' / 'research' / 'vlm_prompting'
assert PKG.exists(), f'Khong thay code tai {PKG}'
sys.path.insert(0, str(PKG))

# Xoa module da import o lan chay truoc, neu khong Python dung ban cu trong bo nho.
for ten in list(sys.modules):
    if ten.startswith(('vlm', 'benchmark_runner', 'checkpoint_utils', 'quality')):
        del sys.modules[ten]

hash_code = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                           cwd=DICH, capture_output=True, text=True).stdout.strip()
print('Code tai:', PKG)
print('Commit:', hash_code)
assert hash_code, 'Khong doc duoc commit hash -- clone that bai'


In [ ]:
ANH_DIR = next(Path('/kaggle/input').glob('**/images'), None)
print('Thu muc anh:', ANH_DIR)
so_anh = len(list(ANH_DIR.glob('*.jpg')))
print('So anh:', so_anh)
assert so_anh >= 100, f'Chi co {so_anh} anh, de bai can >= 100'

In [ ]:
# --strict: model khong tai duoc thi NEM LOI, khong am tham roi ve mock.
# 1B truoc 3B: model nhe chay truoc de co so du phong neu 3B tran bo nho.
# benchmark_runner tu bo qua model nap loi va chay tiep model sau.
lenh = [
    sys.executable, 'scripts/benchmark_runner.py',
    '--mode', 'mass',
    '--models', 'vintern-1b,vintern-3b',
    '--backend', 'transformers',
    '--strict', '--restart',
    '--frames-dir', str(ANH_DIR),
    '--out-dir', '/kaggle/working/ket_qua',
]
print('Chay:', ' '.join(lenh))
kq = subprocess.run(lenh, cwd=str(PKG))
print('Ma thoat:', kq.returncode)
# KHONG assert returncode o day: mot model hong van con model kia, ket qua phai
# duoc nen o cell sau truoc khi bat cu thu gi nem loi.

In [ ]:
import shutil

ZIP = shutil.make_archive('/kaggle/working/benchmark-vintern', 'zip',
                          '/kaggle/working/ket_qua')
print('Da nen:', ZIP)
# Nen TRUOC cell kiem: kernel ERROR thi Kaggle khong luu /kaggle/working,
# mat sach ket qua du benchmark da chay xong.

In [ ]:
import json
from collections import Counter

ra = Path('/kaggle/working/ket_qua')
print('File sinh ra:')
for f in sorted(ra.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(ra)}  {f.stat().st_size:,} bytes')

# Dem theo TUNG model: gop chung thi mot model chi ra vai anh van lot qua nguong
# nho model kia bu vao.
for mk in ('vintern-1b', 'vintern-3b'):
    ck = ra / f'checkpoint_{mk}.json'
    if not ck.exists():
        print(f'\n{mk}: KHONG co checkpoint -- model nay khong nap duoc')
        continue
    d = json.loads(ck.read_text(encoding='utf-8'))
    xong = d.get('da_xong') or {}
    ok = sum(1 for v in xong.values() if v.get('thanh_cong'))
    co_raw = sum(1 for v in xong.values() if v.get('raw_text'))
    print(f'\n{mk}: {len(xong)} anh | JSON hop le {ok} ({ok/max(len(xong),1):.1%})'
          f' | co raw_text {co_raw}')
    # raw_text la bang chung truc tiep model sinh ra gi khi hong -- khong co no
    # thi phai chay lai GPU moi chan doan duoc.
    mau_hong = [v.get('raw_text') for v in xong.values() if v.get('raw_text')][:2]
    for i, t in enumerate(mau_hong, 1):
        print(f'  vi du output hong {i}: {t[:300]}')

kq_file = ra / 'sample_results.json'
if kq_file.exists():
    muc = json.loads(kq_file.read_text(encoding='utf-8'))
    print(f'\nsample_results.json: {len(muc)} muc')
    print('theo model:', Counter(m.get('model') for m in muc))
else:
    print('\nKhong co sample_results.json -- ca hai model deu khong sinh duoc JSON hop le')